# CV-TMLE: cross-fitting and navigator teams

This notebook fits the same average treatment effect as the
[point-treatment tutorial](point-treatment-tmle.ipynb), with flexible learners. Each step shows its
code, its output, and what the output tells you.
[CV-TMLE and cross-fitting](../technical-reference/cv-tmle.md) gives the constructions and their
fold arithmetic.

## The applied question

The point-treatment tutorial fit boosted learners and reported one interval. A reviewer of the
network report asks two questions. The learners predict held-out patients well, so why split the
sample? Patients come from two hundred navigator teams, so are they independent observations?

The question itself does not change. The program sponsor still asks how much the mean 30-day
transition score would change if every eligible discharge received the offer.

## What you will learn

| after this notebook you can | the step that shows it |
| --- | --- |
| state which condition cross-fitting addresses, and which conditions it does not | why this method |
| fit CV-TMLE with an explicit fold count, and read the construction it reports | Step 5 |
| compare a draw on which in-sample nuisance predictions give a narrower interval | Step 6 |
| declare navigator teams, and read what changes in the interval and the folds | Steps 7 and 8 |
| reuse one outer split to compare fits on identical folds | Step 9 |
| tell the three CV-TMLE constructions apart | Step 10 |
| read the diagnostics and the spread across fold draws | Steps 11 and 12 |
| read an omitted-confounding analysis for the cross-fitted estimate | Step 13 |

## Why this method

| your situation | what this method buys | what it costs |
| --- | --- | --- |
| a flexible learner for either nuisance | an interval without a Donsker restriction, under the [remaining conditions](../technical-reference/cv-tmle.md#what-this-solves) | one nuisance fit per outer fold, ten by default |
| patients nested in navigator teams | teams stay intact in every split, and the variance uses team totals | a wider interval, because teams, not patients, are the independent units. With fewer teams than folds, the fold count drops |

Classical interval proofs that reuse observations control an empirical-process term with a
complexity condition such as a Donsker condition. Rich, tuned learners need not satisfy that
condition, even when they predict well. Cross-fitting breaks this data-reuse path because each
patient's nuisance prediction comes from a model that never saw that patient. It addresses no
other condition.

| term | plain meaning |
| --- | --- |
| estimand | the number the question asks for, written before any model is chosen. [Estimands](../user-guide/estimands.md) defines the available questions |
| nuisance | a model the estimate needs but the question does not ask about. Here, the outcome regression Q and the treatment mechanism g. [Point-treatment TMLE](../technical-reference/point-treatment-tmle.md) defines both |
| cross-fitting | each row's nuisance prediction comes from models fit without that row. [CV-TMLE](../technical-reference/cv-tmle.md) defines it |
| targeting | a small update to the outcome model, weighted by the treatment model, that removes first-order bias. [Methods and learners](../user-guide/methods-learners.md#targeting-and-bounds) defines the update |
| outer fold | one part of the sample. The models for its rows train on the other parts |
| influence curve | how much each row moves the estimate. Its variance gives the standard error. [Inference](../technical-reference/inference.md) defines it |
| cluster | a group of rows that are not independent of each other. Here, one navigator team |
| split plan | the realized fold label of every row, which a later fit can reuse |
| calibration slope | the slope of a recalibration of the predicted probabilities. The ideal is 1. A slope below 1 means the predictions are too extreme |
| effective sample size | how many equally weighted rows the weights are worth. A ratio of 25% means the weights are as uneven as if a quarter of the rows counted |

## Step 1: set up

The setup imports the learners and prints the installed `cleverly` version. Every fit below passes
its learners, fold count, and random seed explicitly, so a rerun reproduces the stored outputs.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression

import cleverly

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 20)
print("cleverly", cleverly.__version__)

cleverly 0.1.2


**What this output tells you.** The stored outputs in this notebook came from the version named
above. A different version can print different numbers.

## Step 2: the data

The law is the point-treatment tutorial's law, drawn with a new seed. Its nuisance functions are
nonlinear, which is the case where a flexible learner earns its cost. `navigation_data` draws the
rows under the program's column names. The code prints the column names, the first rows, and the
true ATE.

In [2]:
from cleverly.datasets import navigation_data

frame, truth = navigation_data(n=3_000, seed=34)
print("rows and columns:", frame.shape)
print(list(frame.columns))
print(frame.head().round(3))
print()
print(f"population ATE: {truth['ate']:.3f}")

rows and columns: (3000, 6)
['transition_score', 'transition_navigation', 'discharge_risk', 'prior_utilization', 'medication_burden', 'age']
   transition_score  transition_navigation  discharge_risk  prior_utilization  medication_burden    age
0             1.663                    0.0          -0.040             -1.258              2.574  0.482
1             4.397                    1.0           0.644             -0.208              0.058  0.337
2             3.008                    1.0           0.195             -0.609              0.615  0.490
3            -0.371                    0.0          -0.118             -0.719             -0.879 -1.008
4             4.092                    1.0           0.277             -0.351              0.871 -0.888

population ATE: 1.750


**What this output tells you.** Each row is one discharge. The helper `navigation_data` renames the
columns of the `make_nonlinear_ate` generator and keeps its values. `transition_navigation` is 1 for an
offer and 0 for usual support. The covariates are drawn with mean 0 and SD 1, and the score is in
synthetic units.

The true ATE is 1.750. A real program has no `truth`. Every comparison against it below is a
teaching device.

## Step 3: write the protocol

This page keeps the [shared study design](index.md#the-shared-study-design) and changes only the
estimator. Cross-fitting is an estimation choice, and it must not touch the question. The code
therefore uses `navigation_protocol()` unchanged, as the point-treatment tutorial does.

In [3]:
from cleverly.datasets import navigation_protocol

protocol = navigation_protocol()
print("\n".join(protocol.summary_lines()))

causal study protocol: schema 1; 2dd268e1f5ab29ae
target population: Adults with a discharge-home order at a participating hospital during the enrollment period
eligibility: ['Age 18 years or older', 'Discharge home ordered at a participating hospital']
time zero: Discharge-home order, after baseline measurement and before the navigation offer
treatment strategies: ['Offer standard transition navigation', 'Provide usual discharge support']
treatment versions: ['Bedside transition plan and two scheduled navigator contacts within 30 days', 'No access to the transition-navigation offer']
outcome: Patient-reported transition score
horizon: 30 days after discharge
intercurrent-event handling: ['Use the transition score regardless of readmission', 'Analyze the offer regardless of completed contacts', 'The protocol scores death before day 30 as the worst transition score (composite strategy)']
interference unit: Individual patient
assumption rationale: ['The recorded baseline variables cover 

**What this output tells you.** The first line gives the fingerprint `2dd268e1f5ab29ae`. The
point-treatment tutorial prints the same fingerprint, because the two pages record the same design.
[Point-treatment TMLE](point-treatment-tmle.ipynb) reads each field.

The interference unit stays the individual patient. Shared navigator teams are a statistical
dependence, not interference. `StudyProtocol` has no field for that dependence. Step 7 declares it
through the design's `cluster=` role instead.

Clustering is not an interference adjustment. If patients compete for navigator slots, potential
outcomes can depend on other assignments. A cluster-robust standard error cannot repair that
causal-design failure.

## Step 4: design and identification

The design names the column roles, and the estimand names the contrast. Identification returns the
observed-data formula, the nuisances it needs, and the assumptions that make it causal.

In [4]:
from cleverly import ATE, CausalStudy, PointTreatment

study = CausalStudy(
    frame,
    design=PointTreatment(
        outcome="transition_score",
        treatment="transition_navigation",
        adjustment=("discharge_risk", "prior_utilization", "medication_burden", "age"),
    ),
    protocol=protocol,
)
effect = study.identify(ATE(reference=0))
print(effect.summary())

average treatment effect, E[Y^a] - E[Y^reference]
identified by explicit-adjustment: E_W[E(transition_score | transition_navigation=a, W)] - E_W[E(transition_score | transition_navigation=0, W)] for a in [1]
adjustment/history: ['discharge_risk', 'prior_utilization', 'medication_burden', 'age']
required nuisances: ['outcome_regression', 'treatment_mechanism']
assumptions:
  - consistency: Y = Y^a when A = a
  - no interference: one unit's potential outcome does not depend on other units' treatment assignments
  - no unmeasured confounding: Y^a is independent of A given W
  - positivity: P(transition_navigation = a | W) > 0 almost surely for every supported treatment level a in [0, 1]
causal study protocol: schema 1; 2dd268e1f5ab29ae
target population: Adults with a discharge-home order at a participating hospital during the enrollment period
eligibility: ['Age 18 years or older', 'Discharge home ordered at a participating hospital']
time zero: Discharge-home order, after baseline measu

**What this output tells you.** The estimand, the formula, the two required nuisances, and the four
assumptions match the point-treatment tutorial. The summary also repeats the stored protocol.

Nothing in this output names folds or learners. The fold choice belongs to the method in the next
step, so a change of estimator cannot change the question.

## Step 5: estimate with cross-fitting

Cross-fitting is configured as a named group. The configuration is written out in full.

| setting | value | what it does |
| --- | --- | --- |
| `outcome_learner` | gradient boosting | fits Q, the expected score given the offer and the covariates |
| `treatment_learner` | gradient boosting | fits g, the probability of an offer given the covariates |
| `CrossFitting(n_folds=5)` | five outer folds | predicts each row from models fit on the other four folds |
| `Runtime(random_state=34, n_jobs=1)` | fixed seed, one process | makes the fold draw and the fit reproducible |

The package default is ten outer folds. Five keeps this notebook fast.

The bare boosted learners have no inner learner folds, so `learner_folds` does not apply. The
[methods guide](../user-guide/methods-learners.md#two-fold-layers) explains the two fold layers.

The summary repeats the protocol. Read the construction line, the footer, and the estimate table.

In [5]:
from cleverly import CrossFitting, ModelSpec, Runtime, TMLEMethod

boosted = ModelSpec(
    outcome_learner=HistGradientBoostingRegressor(random_state=34),
    treatment_learner=HistGradientBoostingClassifier(random_state=34),
)
cross_fitted = effect.estimate(
    method=TMLEMethod(
        models=boosted,
        cross_fitting=CrossFitting(n_folds=5),
        runtime=Runtime(random_state=34, n_jobs=1),
    )
)
print(cross_fitted.summary())
print()
print(f"population ATE: {truth['ate']:.3f}")

Targeted maximum likelihood estimation
n = 3000; covariates = 4; P(A=1) = 0.4533
causal estimand: average treatment effect, E[Y^a] - E[Y^reference]
identification: explicit-adjustment; E_W[E(transition_score | transition_navigation=a, W)] - E_W[E(transition_score | transition_navigation=0, W)] for a in [1]
required nuisances: outcome_regression, treatment_mechanism
identification assumptions: consistency: Y = Y^a when A = a; no interference: one unit's potential outcome does not depend on other units' treatment assignments; no unmeasured confounding: Y^a is independent of A given W; positivity: P(transition_navigation = a | W) > 0 almost surely for every supported treatment level a in [0, 1]
causal study protocol: schema 1; 2dd268e1f5ab29ae
target population: Adults with a discharge-home order at a participating hospital during the enrollment period
eligibility: ['Age 18 years or older', 'Discharge home ordered at a participating hospital']
time zero: Discharge-home order, after baseli

**What this output tells you.** The construction line reads
`stacked CV-TMLE (Levy): nuisances cross-fitted over 5 folds; targeting: pooled`. It names the
construction, the number of outer folds, and the targeting scheme. Step 10 compares the other
constructions.

The estimate is 1.694 with a standard error of 0.0995. The 95% interval is [1.499, 1.8891], and it
contains the true ATE of 1.750. That is one draw, not a coverage result.

The footer line records the data fingerprint `1d98be9632f71d93` and the fold fingerprint
`9d73d310c0025a15`. Step 9 uses both.

## Step 6: the failure mode, an interval that is too narrow

Now fit the same learners with the splitting turned off. `CrossFitting(enabled=False)` is not a
tuning knob. It selects ordinary TMLE rather than CV-TMLE. The code prints both fits side by side,
and it marks whether each interval contains the true ATE.

In [6]:
in_sample = effect.estimate(
    method=TMLEMethod(
        models=boosted,
        cross_fitting=CrossFitting(enabled=False),
        runtime=Runtime(random_state=34, n_jobs=1),
    )
)


def summary_line(result, marker):
    """The first line of the result summary that contains ``marker``."""
    return next(line.strip() for line in result.summary().splitlines() if marker in line)


def side_by_side(fits, target=None, columns=("psi", "std_err", "ci_lower", "ci_upper")):
    """One row per labelled fit, and whether each interval contains ``target`` when given."""
    table = pd.concat({label: fitted.to_frame() for label, fitted in fits.items()})
    table = table.droplevel(1)[list(columns)]
    if target is not None:
        table["covers truth"] = table["ci_lower"].le(target) & table["ci_upper"].ge(target)
    return table


print(summary_line(in_sample, ": nuisances "))
comparison = side_by_side(
    {"no cross-fitting": in_sample, "cross-fitted": cross_fitted}, truth["ate"]
)
print(comparison.round(3))
se_ratio = in_sample["ate"].std_error / cross_fitted["ate"].std_error
print(f"in-sample SE / cross-fitted SE: {se_ratio:.2f}")
miss = abs(truth["ate"] - in_sample["ate"].psi) / in_sample["ate"].std_error
print(f"in-sample distance from the truth, in its standard errors: {miss:.1f}")
print(f"population ATE: {truth['ate']:.3f}")

TMLE (in-sample nuisances): nuisances fitted in-sample (cross_fit=False)
                    psi  std_err  ci_lower  ci_upper  covers truth
no cross-fitting  1.677    0.028     1.622     1.732         False
cross-fitted      1.694    0.100     1.499     1.889          True
in-sample SE / cross-fitted SE: 0.28
in-sample distance from the truth, in its standard errors: 2.6
population ATE: 1.750


**What this output tells you.** The first line confirms that the new fit used in-sample nuisances.
The table then compares the two fits on this draw.

| quantity | no cross-fitting | cross-fitted |
| --- | --- | --- |
| point estimate | 1.677 | 1.694 |
| standard error | 0.028 | 0.100 |
| 95% interval | (1.622, 1.732) | (1.499, 1.889) |
| contains the true ATE of 1.750 | no | yes |

The two point estimates are close. The in-sample standard error is 0.28 times the cross-fitted one,
which is less than a third. The in-sample interval misses the true ATE by 2.6 of its standard
errors. A 95% interval misses on about one draw in 20, so one miss can be sampling variation.

On this draw, the boosted in-sample fit gives more even observed-arm weights and a smaller
standard error. Step 11 prints that weight comparison. These values do not isolate the
empirical-process term or prove why the interval narrowed. The
[CV-TMLE reference](../technical-reference/cv-tmle.md#what-this-solves) explains the data-reuse
condition, and the registered study below measures the coverage difference.

One draw does not establish a coverage rate. The registered
[stacked CV-TMLE study](../technical-reference/method-evidence/stacked-point-treatment-cv-tmle.md#theory-properties)
fit this law 400 times at n = 500. It used a fully grown regression tree for Q, a logistic model
for g, and ten folds. The in-sample control covered 65.0% of the time and the cross-fitted fit
89.5%. Both fall below 95%, so the study supports relative recovery, not nominal coverage. The
study's [property results](https://github.com/esbraun/cleverly-tmle/blob/main/tests/canonical/tmle3_cvtmle/properties.csv)
record the design.

## Step 7: the second failure mode, patients are not independent

The program has two hundred navigator teams with fifteen patients each. Teams differ in ways the
recorded covariates do not capture, such as supervisor practice and local follow-up quality. A
shared difference that changes how much navigation helps makes the influence values of one team
move together.

The `cluster=` role declares the teams. A second law carries a shared team effect interacted with
the offer. The two failure modes use separate laws, so the team fit uses linear learners that are
correct for its two covariates.

In this law the team effect changes the benefit but not who receives an offer. A team factor that
also drives offers is an unmeasured confounder, and no cluster declaration repairs it.

The code builds one design without the cluster role and a copy with `cluster="navigator_team"`.
Nothing else differs between the two fits.

In [7]:
from dataclasses import replace

from cleverly.datasets import make_clustered

team_frame, team_truth = make_clustered(n=3_000, seed=34, cluster_size=15)
team_frame = team_frame.rename(
    columns={
        "Y": "transition_score",
        "A": "transition_navigation",
        "W1": "discharge_risk",
        "W2": "medication_burden",
        "cluster": "navigator_team",
    }
)
simple = TMLEMethod(
    models=ModelSpec(
        outcome_learner=LinearRegression(n_jobs=1),
        treatment_learner=LogisticRegression(max_iter=1000, random_state=34),
    ),
    cross_fitting=CrossFitting(n_folds=5),
    runtime=Runtime(random_state=34, n_jobs=1),
)
patients_design = PointTreatment(
    outcome="transition_score",
    treatment="transition_navigation",
    adjustment=("discharge_risk", "medication_burden"),
)
teams_design = replace(patients_design, cluster="navigator_team")

ignoring = (
    CausalStudy(team_frame, design=patients_design, protocol=protocol)
    .identify(ATE(reference=0))
    .estimate(method=simple)
)
clustered = (
    CausalStudy(team_frame, design=teams_design, protocol=protocol)
    .identify(ATE(reference=0))
    .estimate(method=simple)
)

team_comparison = side_by_side(
    {"patients as units": ignoring, "teams as units": clustered}, team_truth["ate"]
)
print(team_comparison.round(3))
team_se_ratio = clustered["ate"].std_error / ignoring["ate"].std_error
print(f"teams SE / patients SE: {team_se_ratio:.2f}")
print("navigator teams:", clustered.data.n_clusters)
print(f"population ATE: {team_truth['ate']:.3f}")

                     psi  std_err  ci_lower  ci_upper  covers truth
patients as units  0.875    0.116     0.648     1.102          True
teams as units     0.880    0.195     0.497     1.262          True
teams SE / patients SE: 1.68
navigator teams: 200
population ATE: 1.000


**What this output tells you.** The table compares one law fitted twice. Only the cluster role
differs.

| quantity | patients as units | teams as units |
| --- | --- | --- |
| point estimate | 0.875 | 0.880 |
| standard error | 0.116 | 0.195 |
| 95% interval | (0.648, 1.102) | (0.497, 1.262) |

The two point estimates are nearly identical. They differ slightly because the team declaration
also regroups the folds. Step 8 shows how. Declaring the navigator team makes the standard error
1.68 times larger. Both intervals contain the true ATE of 1.000 on this draw.

Neither interval is wrong about the estimand. The unclustered one is wrong about how much
information three thousand patients from 200 teams carry. Cluster-robust inference sums the
influence values inside a team, then takes the variance across teams. With singleton teams it
equals the ordinary formula. [Inference](../technical-reference/inference.md#clusters) gives the
formula.

The large-sample argument counts teams, not patients. The interval uses a normal reference with no
small-sample cluster correction, so with few teams it can still be too narrow.

## Step 8: what the team declaration does to the folds

The cluster role also changes how the rows are split. The code reads the realized split plan of the
team fit and counts how many outer folds each team spans. It then fits four teams with five folds
requested, and it prints the warning and the realized fold count.

In [8]:
import warnings

fold_labels = pd.Series(clustered.split_plan.assignments[0], index=team_frame.index)
folds_per_team = fold_labels.groupby(team_frame["navigator_team"]).nunique()
print("largest number of outer folds one team spans:", folds_per_team.max())

four_teams = team_frame["navigator_team"].drop_duplicates().iloc[:4]
few_teams = team_frame[team_frame["navigator_team"].isin(four_teams)].reset_index(drop=True)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    few = (
        CausalStudy(few_teams, design=teams_design, protocol=protocol)
        .identify(ATE(reference=0))
        .estimate(method=simple)
    )
for warning in caught:
    print("warning:", warning.message)
print("rows:", few_teams.shape[0], "teams:", few.data.n_clusters)
print("outer folds requested: 5, realized:", few.split_plan.n_folds)

largest number of outer folds one team spans: 1
rows: 60 teams: 4
outer folds requested: 5, realized: 4


**What this output tells you.** Each team spans at most 1 outer fold. With four teams and five
folds requested, the fit warns and realizes 4 folds.

| what happens | why it matters |
| --- | --- |
| each team lands entirely in one outer fold | a patient's nuisance prediction cannot use outcomes from the patient's own team |
| with fewer teams than folds, the fit reduces the fold count and warns | the number of teams, not the number of patients, bounds the split |

The registered
[clustered CV-TMLE study](../technical-reference/method-evidence/clustered-point-treatment-cv-tmle.md#theory-properties)
compares cluster-robust and independent-row variances on the same influence curves.

## Step 9: reuse the same outer split

Return to the unclustered `cross_fitted` result from Step 5. A second fit can run on exactly its
validation rows. Use this when you compare two methods rather than two splits.

Every point-treatment result exposes its realized outer folds as `split_plan`. Pass that plan back
through `CrossFitting`. The fits below use a different runtime seed, so only the plan can hold the
folds fixed. The last part hands the plan to a new draw of the same size.

In [9]:
from cleverly import DataError

new_seed = Runtime(random_state=35, n_jobs=1)
plan = cross_fitted.split_plan
redrawn = effect.estimate(
    method=TMLEMethod(models=boosted, cross_fitting=CrossFitting(n_folds=5), runtime=new_seed)
)
reused = effect.estimate(
    method=TMLEMethod(
        models=boosted,
        cross_fitting=CrossFitting(n_folds=5, split_plan=plan),
        runtime=new_seed,
    )
)
print(plan)
print("fold fingerprint, Step 5 fit:     ", cross_fitted.provenance.fold_fingerprint)
print("fold fingerprint, new seed:       ", redrawn.provenance.fold_fingerprint)
print("fold fingerprint, new seed + plan:", reused.provenance.fold_fingerprint)
print("same point estimate as Step 5:", reused["ate"].psi == cross_fitted["ate"].psi)
print(
    "same influence curve as Step 5:",
    np.array_equal(reused["ate"].influence_curve, cross_fitted["ate"].influence_curve),
)
reuse_comparison = side_by_side(
    {
        "Step 5 fit": cross_fitted,
        "new seed, new folds": redrawn,
        "new seed, same plan": reused,
    }
)
print(reuse_comparison.round(3))

other_frame, _ = navigation_data(n=3_000, seed=35)
other_study = CausalStudy(other_frame, design=study.design, protocol=protocol)
refusal = None
try:
    other_study.identify(ATE(reference=0)).estimate(
        method=TMLEMethod(
            models=boosted,
            cross_fitting=CrossFitting(n_folds=5, split_plan=plan),
            runtime=new_seed,
        )
    )
except DataError as error:
    refusal = error
    print("refused on other data:", error)

SplitPlan(n=3000, n_folds=5, n_repeats=1, fingerprint=9d73d310c0025a15, source=1d98be9632f71d93)
fold fingerprint, Step 5 fit:      9d73d310c0025a15
fold fingerprint, new seed:        bac8b6ff5b6fed17
fold fingerprint, new seed + plan: 9d73d310c0025a15
same point estimate as Step 5: True
same influence curve as Step 5: True
                       psi  std_err  ci_lower  ci_upper
Step 5 fit           1.694    0.100     1.499     1.889
new seed, new folds  1.655    0.094     1.472     1.839
new seed, same plan  1.694    0.100     1.499     1.889
refused on other data: split plan was realised on data fingerprinting 1d98be9632f71d93, and these data fingerprint cda6083271d930c6. A plan labels rows by position, so a reordering, a replaced column or an added covariate leaves every label pointing at a different unit, and the row count cannot see it. Fit without split_plan= to draw a split for these data, or, when the rows are the same units in the same order, hand over SplitPlan(plan.assignmen

**What this output tells you.** Read the output from the top.

| output line | what it shows |
| --- | --- |
| `SplitPlan(...)` | 3000 rows, 5 folds, 1 repeat, and `source=1d98be9632f71d93`, the data fingerprint from Step 5 |
| fold fingerprints | the new seed alone draws new folds. The plan repeats `9d73d310c0025a15` exactly |
| same point estimate, same influence curve | both `True`. The plan reproduces the Step 5 fit, although the seed changed |
| the table | the new folds move the estimate to 1.655. The reused plan gives 1.694 again |
| `refused on other data` | a `DataError`, because the new draw has a different data fingerprint |

This is reproducible wiring. It adds no evidence about bias, coverage, or efficiency.

A plan labels rows by position, and it records the data it came from. The
[reusable outer split plans](../technical-reference/cv-tmle.md#reusable-outer-split-plans) section
states the contract.

## Step 10: other constructions over the same folds

`cleverly` ships three constructions over these folds. They are different estimators, and each has
its own registered evidence.

| construction | how it is selected | what it does |
| --- | --- | --- |
| [stacked CV-TMLE](../technical-reference/method-evidence/stacked-point-treatment-cv-tmle.md) | `targeting_scheme="pooled"`, the default | stacks all out-of-fold predictions, fits one targeting regression, and evaluates the plug-in on the whole sample |
| [fold-evaluated CV-TMLE](../technical-reference/method-evidence/fold-evaluated-point-treatment-cv-tmle.md) | `CrossFitting(fold_evaluation=True)` | keeps the pooled update, averages the fold plug-ins, and uses a cross-validated variance |
| [fold-targeted CV-TMLE](../technical-reference/method-evidence/fold-targeted-point-treatment-cv-tmle.md) | `CrossFitting(targeting_scheme="fold")` | fits one targeting regression inside each validation fold |

The code fits the two alternatives on the Step 5 plan, so all three use identical folds. It prints
the construction line of each summary and the estimates. The
[variations table](../technical-reference/cv-tmle.md#variations) lists what each one refuses.

In [10]:
alternatives = {
    "fold-evaluated": CrossFitting(n_folds=5, fold_evaluation=True, split_plan=plan),
    "fold-targeted": CrossFitting(n_folds=5, targeting_scheme="fold", split_plan=plan),
}
construction_fits = {"stacked": cross_fitted}
for label, folds in alternatives.items():
    construction_fits[label] = effect.estimate(
        method=TMLEMethod(
            models=boosted, cross_fitting=folds, runtime=Runtime(random_state=34, n_jobs=1)
        )
    )
for label, fitted in construction_fits.items():
    print(f"{label}: {summary_line(fitted, ': nuisances ')}")
construction_comparison = side_by_side(construction_fits)
print(construction_comparison.round(4))

stacked: stacked CV-TMLE (Levy): nuisances cross-fitted over 5 folds; targeting: pooled
fold-evaluated: fold-evaluated CV-TMLE: nuisances cross-fitted over 5 folds; targeting: pooled
fold-targeted: fold-specific targeted TMLE: nuisances cross-fitted over 5 folds; targeting: fold
                   psi  std_err  ci_lower  ci_upper
stacked         1.6940   0.0995    1.4990    1.8891
fold-evaluated  1.6940   0.0995    1.4989    1.8892
fold-targeted   1.6682   0.0755    1.5202    1.8161


**What this output tells you.** Each summary names its own construction. The fold-targeted fit calls
itself `fold-specific targeted TMLE`, with `targeting: fold`.

| construction | estimate | standard error |
| --- | --- | --- |
| stacked | 1.6940 | 0.0995 |
| fold-evaluated | 1.6940 | 0.0995 |
| fold-targeted | 1.6682 | 0.0755 |

The fold-evaluated fit keeps the stacked update and averages the plug-ins of equal-size folds. Its
point estimate therefore equals the stacked one. Only its variance formula differs, and the
interval endpoints differ in the fourth decimal. The fold-targeted estimate and standard error
differ more.

One draw does not rank the three estimators. A smaller standard error here is not evidence of
better coverage. Each construction's evidence page reports its own repeated-sampling results.

## Step 11: diagnostics, what the fit can check

Start with the combined assessment of the Step 5 fit. It presents validation, diagnostics, and
sensitivity together. `assessment.attention` lists the rows that need review. The code then prints
three retained reports: support, nuisance models, and score equations. The last line gives the
support detail of the in-sample fit for comparison.

In [11]:
assessment = cross_fitted.assess()
print(assessment.summary())
print("needs attention:", tuple(item.name for item in assessment.attention))
support = assessment.report("support")
nuisance = assessment.report("nuisance_models")
scores = assessment.report("score_equations")
print()
print(support.summary())
print()
print(nuisance.summary())
print()
print(scores.summary())
print()
smallest = support.propensity_quantiles["overall"][0.0]
print(f"smallest estimated g(W), before truncation: {smallest:.4f}")
truncated = support.truncated
print(f"truncated: {truncated['count']:.0f} units ({truncated['fraction']:.2%})")
for arm, ess in support.effective_sample_size.items():
    ratio = f"{ess['effective']:.1f} ({ess['ratio']:.1%})"
    print(f"{arm}: {ess['n']:.0f} rows, effective sample size {ratio}")
print()
print("in-sample support:", in_sample.diagnostics.run_all()["support"].detail)

Returned results
----------------
surface      operation            result                                                                                                                                                                                   
-----------  -------------------  -----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
validation   support              maximum truncated fraction 1.0%; minimum effective-sample-size ratio 25.2%; group load: mean:h1 342.9/3000 Kish-equivalent mask rows (11.4%; 11.4% all; draw 01 of 01); not estimator ESS
sensitivity  omitted_confounding  at the default strengths; cf_y=0.03, cf_d=0.03, rho=1; bias-adjusted interval [1.533, 1.856]                                                                                             
sensitivity  robustness_value     point robustness value 0.2726; confidence-limit valu

**What this output tells you.** Read the parts in order.

| output part | what it shows on this draw |
| --- | --- |
| `Checks` and `needs attention` | one warning, `nuisance_models`. The score-equation check passed |
| nuisance model diagnostics | the out-of-fold propensity is poorly calibrated, with a slope of 0.49 against an ideal of 1.0 |
| support fields | the smallest estimated g(W) is 0.0011. 30 units (1.00%) sit at the truncation bound. The treated arm has an effective sample size of 25.2% of its rows |
| in-sample support | the in-sample fit reports a minimum effective-sample-size ratio of 91.6% |

The `Returned results` rows are calculations that ran. They are not passes. Step 13 reads the
sensitivity rows. The nuisance report measures the models on patients they did not see.

On this draw, the slope of 0.49 says that the fitted propensities are more extreme than the
observed offer rates under this recalibration summary. The small g(W), the truncated units, and
the treated-arm ratio of 25.2% also show concentrated weights on this draw. Review the model and
the support report together before changing the learner.

`cleverly` derives no threshold for the effective sample size, so the support report gives 25.2% no
grade.

The in-sample ratio is much higher, although the true assignment mechanism is the same. On this
draw, its observed-arm weights are closer to one. Cross-fitting did not cause the support pattern,
and it does not repair it. The 25.2% describes the estimated mechanism, not the population.

Positivity means that every kind of patient has some chance of each arm. The
[diagnostics guide](../user-guide/results-assessment.md#diagnostics) describes each report.

## Step 12: the fold draw itself

One split is one draw. `repeats=` runs the complete estimator on several fold draws and reports the
median. With several estimates, a repeated fit refuses a simultaneous band. This fit reports one
estimate, so the call keeps the default `Inference()`.

In [12]:
repeated = effect.estimate(
    method=TMLEMethod(
        models=boosted,
        cross_fitting=CrossFitting(n_folds=5, repeats=3),
        runtime=Runtime(random_state=34, n_jobs=1),
    )
)
print(summary_line(repeated, "median over"))
print(side_by_side({"median of 3 draws": repeated}).round(3))
repeated_nuisance = repeated.diagnostics.run_all().report("nuisance_models")
print(repeated_nuisance.repeat_spread_frame().round(4).to_string(index=False))

(median over 3 independent draws of the split; variance includes within-draw uncertainty and split dispersion)
                     psi  std_err  ci_lower  ci_upper
median of 3 draws  1.696    0.093     1.513     1.879
estimand  n_repeats  standard_deviation  reported_standard_error  ratio_to_standard_error
     ate          3              0.0076                   0.0932                   0.0814


**What this output tells you.** The first line confirms the median over 3 independent draws. The
median estimate is 1.696 with a standard error of 0.093.

`repeats=` is the same estimator over several draws, not a new estimator. It reports the median
point, and its variance includes the split dispersion. The
[variations table](../technical-reference/cv-tmle.md#variations) states what a repeated fit refuses.

The reported standard error of 0.093 is below the Step 5 value of 0.0995. The fold draw also moves
the standard error. In Step 9, a new draw gave 0.094 against 0.100. The median rule takes the
median over the draws of each variance plus the squared distance from the median point. The result
can therefore fall below the value of one draw.

The spread row gives a standard deviation of 0.0076 across the three draws. The reported standard
error is 0.0932, and their ratio is 0.0814. Across these three draws, the estimate moved much less
than its standard error. Three draws give an imprecise spread. In Step 9, one new draw moved the
estimate from 1.694 to 1.655, about five times this standard deviation. The report defines no
threshold for the ratio.

The [repeated CV-TMLE evidence](../technical-reference/method-evidence/repeated-cross-fitting.md)
gives the repeated-sampling results for the median report.

## Step 13: sensitivity, what folds cannot check

Sensitivity analysis asks how strong an unmeasured confounder would need to be to change the
conclusion. Cross-fitting changes the estimator, but it does not change the
no-unmeasured-confounding assumption. The assessment already holds the omitted-confounding bound
and robustness value for this cross-fitted ATE.

| term | plain meaning |
| --- | --- |
| `cf_y` | the share of the remaining outcome variation that a hidden confounder explains |
| `cf_d` | the share of the remaining treatment variation that a hidden confounder explains |
| `rho` | how closely the confounder's two effects align. `rho=1` is the worst case |
| robustness value | the equal `cf_y` and `cf_d` that moves the estimate to zero at `rho=1` |

The [sensitivity guide](../user-guide/results-assessment.md#sensitivity-analysis) defines these
quantities. The point-treatment tutorial shows how to benchmark them against a recorded covariate.

In [13]:
robustness = assessment.report("robustness_value")
confounding = assessment.report("omitted_confounding")
print(f"robustness value: {robustness['rv']:.3f} (confidence-limit value {robustness['rva']:.3f})")
print(
    f"default strengths: cf_y={confounding.cf_y:.2f}, "
    f"cf_d={confounding.cf_d:.2f}, rho={confounding.rho:.0f}"
)
print(f"bias-adjusted bounds: ({confounding.lower:.3f}, {confounding.upper:.3f})")
print(f"one-sided 95% confidence limits: ({confounding.ci_lower:.3f}, {confounding.ci_upper:.3f})")

robustness value: 0.273 (confidence-limit value 0.238)
default strengths: cf_y=0.03, cf_d=0.03, rho=1
bias-adjusted bounds: (1.533, 1.856)
one-sided 95% confidence limits: (1.368, 2.021)


**What this output tells you.** The point robustness value is 0.273. At `rho=1`, equal strengths
of 0.273 move the point estimate to zero. The confidence-limit value is 0.238. It is the equal
strength that moves the confidence limit, not the point estimate, to zero.

At the default strengths of 0.03, the bias-adjusted bounds are (1.533, 1.856). The one-sided 95%
confidence limits are (1.368, 2.021), so both remain above zero on this analysis. These numbers
are conditional on the omitted-variable model and its worst-case alignment.

Cross-fitting does not make sensitivity analysis unnecessary. It answers an identification
question that the fold construction cannot check. The
[point-treatment tutorial](point-treatment-tmle.ipynb#step-10-sensitivity-what-the-fit-cannot-check)
benchmarks the same quantities against `discharge_risk`.

## How far to trust this

| reviewer question | answer on this page |
| --- | --- |
| why split the sample | without splitting, the boosted fit reported a standard error 0.28 times the cross-fitted one (Step 6). The registered study shows that in-sample intervals cover less often on this law |
| are patients independent | no. The team law gives each team a shared effect. Declaring teams made the standard error 1.68 times larger and kept each team in one fold (Steps 7 and 8) |

| layer | establishes | does not establish |
| --- | --- | --- |
| the two comparisons in Steps 6 and 7 | that in-sample nuisances and undeclared teams each gave a narrower interval on this draw | the coverage rate of any of these estimators |
| the out-of-fold nuisance report | how the learners performed on unseen patients | that the learners converge fast enough for the remainder condition |
| the retained support report | truncation and effective sample size | that cross-fitting repairs poor support. It does not |
| the retained split-spread rows | how much the estimate moved across the three declared fold draws | whether that amount is acceptable or another draw would help |
| the omitted-confounding analysis | the bound under its declared strength and alignment | that no stronger hidden confounder exists |
| the registered studies | each construction passes its accuracy and property rows, within the limits its page declares | that your identification assumptions hold on your data, or coverage for learners and fold counts no row covers |

No registered row covers five boosted folds exactly. The
[technical entry](../technical-reference/cv-tmle.md#validation-issues-special-to-this-method)
links the evidence for each construction.

## Where to go next

Cross-fitting addresses only the Donsker condition. If you doubt one nuisance and still want an
interval, the condition you are worried about is the
[product rate](../technical-reference/cv-tmle.md#what-this-solves). The variant for it is
[DR-TMLE](dr-tmle.ipynb). If your worry is instead which baseline variables belong in the assignment
model, read [collaborative TMLE](collaborative-tmle.ipynb).

The [examples index](index.md#the-program) lists every tutorial in the program.